# Dataset static analysis

This notebook audits the dataset manifests without loading model tensors. It covers class balance, split composition, video-level distributions, clips per video, leakage and duplicate checks, balanced class weights, and an optional file inventory.

Run all cells from the repository root or directly from the `notebooks/` directory.

In [1]:
from pathlib import Path
import html

import pandas as pd

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = str
    display = print

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")


def find_repo_root(start: Path | None = None) -> Path:
    """Locate the project root from a starting directory.

    Args:
        start: Directory from which to begin the upward search. Defaults to
            the current working directory.

    Returns:
        Repository root containing the `configs` and `src` directories.

    Raises:
        FileNotFoundError: If no repository root can be found.
    """
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "configs").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the repository root")


REPO_ROOT = find_repo_root()
MANIFEST_DIR = REPO_ROOT / "data" / "manifests"
SPLIT_ORDER = ["train", "val", "test"]
LABEL_ORDER = ["real", "fake"]
COLORS = {"real": "#2563eb", "fake": "#dc2626"}

print(f"Repository: {REPO_ROOT}")
print(f"Manifests:  {MANIFEST_DIR}")

Repository: /home/infres/mvo-23/projects/fgi-deepfake-detection-pytorch
Manifests:  /home/infres/mvo-23/projects/fgi-deepfake-detection-pytorch/data/manifests


In [2]:
def grouped_bar_chart(data: pd.DataFrame, title: str, y_label: str):
    """Render a dependency-free grouped bar chart as inline SVG.

    Args:
        data: Table whose index defines groups and columns define series.
        title: Chart title displayed above the plot.
        y_label: Label displayed along the vertical axis.

    Returns:
        Display-ready HTML object containing the generated SVG.
    """
    width, height = 760, 390
    left, right, top, bottom = 75, 25, 55, 75
    plot_width = width - left - right
    plot_height = height - top - bottom
    maximum = max(float(data.to_numpy().max()), 1.0)
    groups = list(data.index)
    series = list(data.columns)
    group_width = plot_width / max(len(groups), 1)
    bar_width = group_width * 0.7 / max(len(series), 1)
    elements = [
        f'<svg viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg">',
        '<style>text { font-family: sans-serif; fill: #1f2937; }</style>',
        f'<text x="{width / 2}" y="28" text-anchor="middle" font-size="19" font-weight="600">{html.escape(title)}</text>',
        f'<line x1="{left}" y1="{top + plot_height}" x2="{left + plot_width}" y2="{top + plot_height}" stroke="#6b7280"/>',
        f'<line x1="{left}" y1="{top}" x2="{left}" y2="{top + plot_height}" stroke="#6b7280"/>',
    ]
    for tick in range(5):
        value = maximum * tick / 4
        y = top + plot_height - plot_height * tick / 4
        elements.append(f'<line x1="{left}" y1="{y:.1f}" x2="{left + plot_width}" y2="{y:.1f}" stroke="#e5e7eb"/>')
        elements.append(f'<text x="{left - 10}" y="{y + 4:.1f}" text-anchor="end" font-size="11">{value:,.0f}</text>')
    for group_index, group in enumerate(groups):
        group_start = left + group_index * group_width + group_width * 0.15
        for series_index, name in enumerate(series):
            value = float(data.loc[group, name])
            bar_height = plot_height * value / maximum
            x = group_start + series_index * bar_width
            y = top + plot_height - bar_height
            color = COLORS.get(str(name), "#64748b")
            elements.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar_width * 0.88:.1f}" height="{bar_height:.1f}" fill="{color}" rx="2"/>')
            elements.append(f'<text x="{x + bar_width * 0.44:.1f}" y="{max(y - 5, top + 10):.1f}" text-anchor="middle" font-size="10">{value:,.0f}</text>')
        elements.append(f'<text x="{left + (group_index + 0.5) * group_width:.1f}" y="{top + plot_height + 25}" text-anchor="middle" font-size="12">{html.escape(str(group))}</text>')
    legend_x = left + plot_width - len(series) * 100
    for index, name in enumerate(series):
        x = legend_x + index * 100
        color = COLORS.get(str(name), "#64748b")
        elements.append(f'<rect x="{x}" y="{height - 25}" width="12" height="12" fill="{color}"/>')
        elements.append(f'<text x="{x + 18}" y="{height - 15}" font-size="11">{html.escape(str(name))}</text>')
    elements.append(f'<text x="18" y="{top + plot_height / 2}" text-anchor="middle" font-size="12" transform="rotate(-90 18 {top + plot_height / 2})">{html.escape(y_label)}</text>')
    elements.append("</svg>")
    return HTML("".join(elements))

## Load and validate manifests

In [3]:
required_columns = {"clip_path", "label", "video_id", "clip_id"}
manifest_frames = []

for split in SPLIT_ORDER:
    manifest_path = MANIFEST_DIR / f"{split}_manifest.csv"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Missing manifest: {manifest_path}")
    frame = pd.read_csv(
        manifest_path,
        dtype={"clip_path": str, "label": str, "video_id": str, "clip_id": str},
    )
    missing_columns = required_columns - set(frame.columns)
    if missing_columns:
        raise ValueError(f"{manifest_path.name} is missing {sorted(missing_columns)}")
    frame["manifest_split"] = split
    manifest_frames.append(frame)

samples = pd.concat(manifest_frames, ignore_index=True)
samples["clip_path_resolved"] = samples["clip_path"].map(
    lambda value: Path(value) if Path(value).is_absolute() else REPO_ROOT / value
)

print(f"Loaded {len(samples):,} clips from {len(manifest_frames)} manifests")
display(samples.head())

Loaded 4,000 clips from 3 manifests


,clip_path,label,video_id,clip_id,video_id_hashmod10,split,manifest_split,clip_path_resolved
0,data/processed/train/real/agrmhtjdlk/clips/000000,real,agrmhtjdlk,000000,6,train,train,/home/infres/mvo-23/projects/fgi-deepfake-dete...
1,data/processed/train/real/agrmhtjdlk/clips/000001,real,agrmhtjdlk,000001,6,train,train,/home/infres/mvo-23/projects/fgi-deepfake-dete...
2,data/processed/train/real/agrmhtjdlk/clips/000002,real,agrmhtjdlk,000002,6,train,train,/home/infres/mvo-23/projects/fgi-deepfake-dete...
3,data/processed/train/real/agrmhtjdlk/clips/000003,real,agrmhtjdlk,000003,6,train,train,/home/infres/mvo-23/projects/fgi-deepfake-dete...
4,data/processed/train/real/agrmhtjdlk/clips/000004,real,agrmhtjdlk,000004,6,train,train,/home/infres/mvo-23/projects/fgi-deepfake-dete...


## Class and split balance

Clip-level balance is useful for the loss, while video-level balance reveals whether a few source videos generate a disproportionate number of clips.

In [4]:
clip_counts = (
    samples.groupby(["manifest_split", "label"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(index=SPLIT_ORDER, columns=LABEL_ORDER, fill_value=0)
)
clip_percentages = clip_counts.div(clip_counts.sum(axis=1), axis=0).mul(100)

clip_balance = pd.concat(
    {"count": clip_counts, "percent": clip_percentages}, axis=1
)
display(clip_balance)
display(grouped_bar_chart(clip_counts, "Clip count by split and class", "Clips"))

count       percent       
label           real  fake    real   fake
manifest_split                           
train            540  2260  19.286 80.714
val              180   540  25.000 75.000
test              50   430  10.417 89.583

In [5]:
video_counts = (
    samples.groupby(["manifest_split", "label"], observed=True)["video_id"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(index=SPLIT_ORDER, columns=LABEL_ORDER, fill_value=0)
)
video_percentages = video_counts.div(video_counts.sum(axis=1), axis=0).mul(100)

video_balance = pd.concat(
    {"count": video_counts, "percent": video_percentages}, axis=1
)
display(video_balance)
display(grouped_bar_chart(video_counts, "Unique videos by split and class", "Videos"))

count      percent       
label           real fake    real   fake
manifest_split                          
train             54  226  19.286 80.714
val               18   54  25.000 75.000
test               5   43  10.417 89.583

## Clips per video distribution

In [6]:
clips_per_video = (
    samples.groupby(["manifest_split", "label", "video_id"], observed=True)
    .size()
    .rename("num_clips")
    .reset_index()
)

clips_per_video_summary = (
    clips_per_video.groupby(["manifest_split", "label"], observed=True)["num_clips"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])
    .round(2)
)
display(clips_per_video_summary)

largest_sources = clips_per_video.nlargest(15, "num_clips")
display(largest_sources)

count   mean   std    min    25%    50%    75%    90%  \
manifest_split label                                                           
test           fake   43.000 10.000 0.000 10.000 10.000 10.000 10.000 10.000   
               real    5.000 10.000 0.000 10.000 10.000 10.000 10.000 10.000   
train          fake  226.000 10.000 0.000 10.000 10.000 10.000 10.000 10.000   
               real   54.000 10.000 0.000 10.000 10.000 10.000 10.000 10.000   
val            fake   54.000 10.000 0.000 10.000 10.000 10.000 10.000 10.000   
               real   18.000 10.000 0.000 10.000 10.000 10.000 10.000 10.000   

                        95%    max  
manifest_split label                
test           fake  10.000 10.000  
               real  10.000 10.000  
train          fake  10.000 10.000  
               real  10.000 10.000  
val            fake  10.000 10.000  
               real  10.000 10.000

,manifest_split,label,video_id,num_clips
0,test,fake,aettqgevhz,10
1,test,fake,aevrfsexku,10
2,test,fake,agqphdxmwt,10
3,test,fake,ahbweevwpv,10
4,test,fake,altziddtxi,10
5,test,fake,aneclqfpbt,10
6,test,fake,aqpnvjhuzw,10
7,test,fake,arlmiizoob,10
8,test,fake,asdpeebotb,10
9,test,fake,avtycwsgyb,10


## Integrity and leakage checks

A source video must appear in exactly one split and keep one label. Duplicate `(video_id, clip_id)` pairs or missing clip directories indicate manifest problems.

In [7]:
videos_across_splits = samples.groupby("video_id")["manifest_split"].nunique()
video_label_counts = samples.groupby("video_id")["label"].nunique()
duplicate_clip_rows = samples.duplicated(["video_id", "clip_id"], keep=False)
missing_clip_paths = ~samples["clip_path_resolved"].map(Path.is_dir)
invalid_labels = ~samples["label"].isin(LABEL_ORDER)

if "split" in samples.columns:
    split_mismatches = samples["split"].ne(samples["manifest_split"])
else:
    split_mismatches = pd.Series(False, index=samples.index)

quality_checks = pd.DataFrame(
    {
        "check": [
            "videos present in multiple splits",
            "videos associated with multiple labels",
            "rows in duplicated video_id/clip_id groups",
            "missing clip directories",
            "invalid labels",
            "manifest/declaration split mismatches",
        ],
        "issues": [
            int(videos_across_splits.gt(1).sum()),
            int(video_label_counts.gt(1).sum()),
            int(duplicate_clip_rows.sum()),
            int(missing_clip_paths.sum()),
            int(invalid_labels.sum()),
            int(split_mismatches.sum()),
        ],
    }
)
quality_checks["status"] = quality_checks["issues"].map(
    lambda count: "PASS" if count == 0 else "CHECK"
)
display(quality_checks)

,check,issues,status
0,videos present in multiple splits,0,PASS
1,videos associated with multiple labels,0,PASS
2,rows in duplicated video_id/clip_id groups,0,PASS
3,missing clip directories,0,PASS
4,invalid labels,0,PASS
5,manifest/declaration split mismatches,0,PASS


## Training class weights

For weighted cross entropy, the project uses `weight[c] = N / (K * count[c])`, calculated from the training split only.

In [8]:
train_counts = clip_counts.loc["train"].astype(float)
num_train_samples = train_counts.sum()
num_classes = len(train_counts)
balanced_weights = num_train_samples / (num_classes * train_counts)

class_weight_table = pd.DataFrame(
    {
        "train_clips": train_counts.astype(int),
        "train_percent": train_counts.div(num_train_samples).mul(100),
        "balanced_weight": balanced_weights,
    }
)
display(class_weight_table)

,train_clips,train_percent,balanced_weight
label,,,
real,540,19.286,2.593
fake,2260,80.714,0.619


## Optional clip file inventory

Set `SCAN_CLIP_FILES = True` to inspect the number of frame files and `audio.wav` sizes. This touches every clip directory, so it is kept optional for network filesystems.

In [9]:
SCAN_CLIP_FILES = False

if SCAN_CLIP_FILES:
    def inspect_clip(clip_path: Path) -> pd.Series:
        """Collect lightweight file statistics for one clip directory.

        Args:
            clip_path: Directory containing frame images and `audio.wav`.

        Returns:
            Series containing the frame count, audio presence, and audio
            file size in KiB.
        """
        frame_files = list(clip_path.glob("*.jpg")) + list(clip_path.glob("*.png"))
        audio_path = clip_path / "audio.wav"
        return pd.Series(
            {
                "num_frames": len(frame_files),
                "has_audio": audio_path.is_file(),
                "audio_size_kib": audio_path.stat().st_size / 1024 if audio_path.is_file() else 0.0,
            }
        )

    file_inventory = samples["clip_path_resolved"].apply(inspect_clip)
    samples_with_files = pd.concat([samples, file_inventory], axis=1)
    display(
        samples_with_files.groupby(["manifest_split", "label"], observed=True)[
            ["num_frames", "audio_size_kib"]
        ].describe()
    )
    print(f"Clips without audio.wav: {(~samples_with_files['has_audio']).sum():,}")
    print(f"Clips without frames:    {(samples_with_files['num_frames'] == 0).sum():,}")
else:
    print("File inventory skipped. Set SCAN_CLIP_FILES = True to enable it.")

File inventory skipped. Set SCAN_CLIP_FILES = True to enable it.


## Compact findings

In [10]:
overall_counts = samples["label"].value_counts().reindex(LABEL_ORDER, fill_value=0)
majority_label = overall_counts.idxmax()
minority_label = overall_counts.idxmin()
imbalance_ratio = overall_counts.max() / overall_counts.min()

print(f"Total clips: {len(samples):,}")
print(f"Total unique videos: {samples['video_id'].nunique():,}")
print(
    f"Overall imbalance: {majority_label}/{minority_label} = "
    f"{imbalance_ratio:.2f}:1"
)
print(
    "Training weights [real, fake]: "
    f"[{balanced_weights['real']:.3f}, {balanced_weights['fake']:.3f}]"
)
print(f"Integrity issues detected: {quality_checks['issues'].sum():,}")

Total clips: 4,000
Total unique videos: 400
Overall imbalance: fake/real = 4.19:1
Training weights [real, fake]: [2.593, 0.619]
Integrity issues detected: 0
